# Lesson 3.3 - From bytes to rows

The first How beat of Module 3 (Ingest), run by hand on the kit's own files and functions. It proves what the lesson page reads:

- the message contract (`services/ingest/contracts.py`) refuses a path that escapes its prefix and an object with no tenant folder, and names the field;
- the chunker (`shared/documind_corpus.py`, the same rules as the worker's `_chunk`) mints 283 chunks for both revisions of the ACME handbook, and a re-wrapped paragraph keeps its hash;
- a one-clause edit embeds 2 chunks and reuses 281 (`plan_carry_over`, copied from `indexer.py`);
- the embedding batcher (`batches`, copied from `indexer.py`) needs 4 calls for the handbook and 47 for the whole ACME corpus;
- the sparse encoder (`shared/sparse_encoder.py`) gives one chunk 33 dimensions.

**Cost.** Every cell up to the last one is Rs 0: pure Python on the cloned kit, no Google call. The last cell embeds three chunks with `text-embedding-005`, costs a fraction of a rupee, needs a project you can bill, and is marked `# COSTS MONEY`. Skip it and nothing else changes.

## Install

Only what the cells import: `pydantic` for the contract, `google-genai` for the last cell. The kit's chunker and sparse encoder need nothing.

In [ ]:
%pip install -q pydantic google-genai

## The kit

The public learner repo mirrors the kit under `deploy/` (the course repo renamed its own copy `code/` on 21 September 2026; the cell tries both). The kit root goes on the path so `shared.*` imports, and `services/ingest` so `contracts` imports. Never `import indexer` or `import main`: both build cloud clients at import time. Their pure functions are copied below, with the line they come from.

In [ ]:
import os, subprocess, sys
KIT_BRANCH = "main"
if not os.path.isdir("agentic-ai-weekend-gcp-learners"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", KIT_BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners"], check=True)
KIT = os.path.abspath(next(p for p in ("agentic-ai-weekend-gcp-learners/deploy",
                                       "agentic-ai-weekend-gcp-learners/code") if os.path.isdir(p)))
for p in (KIT, os.path.join(KIT, "services", "ingest")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("kit:", KIT)
print(sorted(f for f in os.listdir(os.path.join(KIT, "services", "ingest")) if f.endswith(".py")))

## Auth

Only the last cell touches the project. Set `PROJECT_ID` to your own project; on Colab this opens the sign-in prompt, elsewhere it does nothing and Application Default Credentials (`gcloud auth application-default login`) are used. Never an API key.

In [ ]:
PROJECT_ID = "documind-ai-YOUR-ID"
try:
    from google.colab import auth; auth.authenticate_user()
except ImportError:
    pass   # not on Colab: use `gcloud auth application-default login`

## 1. The handbook through the chunker (Rs 0)

`chunk_document` is the notebooks' copy of the worker's two rules: one chunk per `## ` section, keyed by the clause code in its heading, and 2,000-character windows with a 200 overlap for everything else. Each chunk carries a `locator` and a `chunk_hash`.

In [ ]:
from shared import documind_corpus as dc

HANDBOOK = os.path.join(KIT, "evals", "corpus", "acme", "hr_policy_2026.md")
bytes_v1 = open(HANDBOOK, "rb").read()
text_v1 = bytes_v1.decode("utf-8")
doc_v1 = {"slug": "hr_policy_2026", "doc_type": "policy",
          "source_uri": "gs://documind-ai-YOUR-ID-uploads/acme/hr_policy_2026.md", "text": text_v1}
chunks_v1 = dc.chunk_document(doc_v1, "acme")
print(len(chunks_v1), "chunks from", f"{len(text_v1):,}", "characters and", text_v1.count("\n## "), "headings")
for c in chunks_v1[:5]:
    print(f"{c['locator']:10} {c['chunk_hash'][:12]}  {len(c['text']):4} chars  {c['chunk_id']}")

Expected: `283 chunks`; the first five locators are `preamble`, `NP-03`, `PB-02`, `LV-01`, `LV-07`, and NP-03's hash starts `f4512754ae41`.

## 2. A re-wrapped paragraph is the same paragraph (Rs 0)

`chunk_hash` (the worker's, in `contracts.py`, and the notebooks' copy) hashes the text with every run of whitespace collapsed to one space. Different line breaks, same hash; one figure changed, a new chunk.

In [ ]:
import re
from contracts import chunk_hash            # services/ingest/contracts.py, the worker's rule

np03 = next(c for c in chunks_v1 if c["locator"] == "NP-03")
rewrapped = re.sub(r"\n", " ", np03["text"])          # every line break becomes a space
indented = np03["text"].replace("\n", "\n    ")         # an editor that indents continuation lines
print("worker  :", chunk_hash(np03["text"])[:16], chunk_hash(rewrapped)[:16], chunk_hash(indented)[:16])
print("notebook:", dc.chunk_hash(np03["text"])[:16], dc.chunk_hash(rewrapped)[:16], dc.chunk_hash(indented)[:16])
assert chunk_hash(np03["text"]) == chunk_hash(rewrapped) == chunk_hash(indented) == dc.chunk_hash(np03["text"])
changed = np03["text"].replace("60 days", "90 days")
print("one figure changed:", chunk_hash(changed)[:16], "-> a new chunk")

## 3. One clause edited: the carry-over plan (Rs 0)

`plan_carry_over` is pure, so it is copied from `services/ingest/indexer.py` (line 94). `held` stands in for what `held_vectors()` returns on the lane: the current rows' vectors, keyed by hash. The demo's revision 2 changes NP-03 (60 to 90 days) and adds a title line and an effective date above the first heading.

In [ ]:
# services/ingest/indexer.py, line 94 - copied, because indexer.py builds a genai client at import
def plan_carry_over(chunks: list[dict], held: dict[str, list[float]]) -> tuple[list, list[int]]:
    vectors, misses = [], []
    for index, chunk in enumerate(chunks):
        vector = held.get(chunk.get("chunk_hash") or "")
        vectors.append(vector)
        if vector is None:
            misses.append(index)
    return vectors, misses

held = {c["chunk_hash"]: [0.0] * 768 for c in chunks_v1}     # what held_vectors() would return: current rows' vectors, by hash

text_v2 = open(os.path.join(KIT, "evals", "demo", "hr_policy_2026_v2.md"), encoding="utf-8").read()
chunks_v2 = dc.chunk_document(dict(doc_v1, text=text_v2), "acme")
vectors_v2, misses_v2 = plan_carry_over(chunks_v2, held)
print(f"revision 2: {len(chunks_v2)} chunks, reused={len(chunks_v2) - len(misses_v2)} embedded={len(misses_v2)}",
      [chunks_v2[i]["locator"] for i in misses_v2])

edited = dc.chunk_document(dict(doc_v1, text=text_v1.replace("notice period of 60 days", "notice period of 90 days")), "acme")
_, misses_edit = plan_carry_over(edited, held)
print(f"one clause by hand: {len(edited)} chunks, reused={len(edited) - len(misses_edit)} embedded={len(misses_edit)}",
      [edited[i]["locator"] for i in misses_edit])

Expected: `revision 2: 283 chunks, reused=281 embedded=2 ['preamble', 'NP-03']` and, for the hand edit that touches nothing above the first heading, `reused=282 embedded=1 ['NP-03']`. On the lane the same two numbers are on the `ingest_ok` line: `reused=281 embedded=2 retired=283`.

## 4. How many embedding calls (Rs 0)

`batches` is copied from `indexer.py` (line 35) with its constants (lines 19 to 21): at most 250 texts and about 15,000 estimated tokens a request, three characters a token. Then the same count over every ACME document `load_documents` finds.

In [ ]:
# services/ingest/indexer.py, lines 19-21 and 35 - copied
EMBED_BATCH = 250
EMBED_TOKENS = 15_000
CHARS_PER_TOKEN = 3

def batches(texts: list[str]) -> list[list[str]]:
    out, cur, cur_tokens = [], [], 0
    for text in texts:
        tokens = max(1, len(text) // CHARS_PER_TOKEN)
        if cur and (len(cur) >= EMBED_BATCH or cur_tokens + tokens > EMBED_TOKENS):
            out.append(cur)
            cur, cur_tokens = [], 0
        cur.append(text)
        cur_tokens += tokens
    if cur:
        out.append(cur)
    return out

b = batches([c["text"] for c in chunks_v1])
print("handbook, first load:", len(b), "calls; texts per call", [len(x) for x in b])
print("handbook, re-issue  :", len(batches([chunks_v2[i]["text"] for i in misses_v2])), "call for", len(misses_v2), "texts")

evals_dir = os.path.join(KIT, "evals")
corpus = []
for d in dc.load_documents("acme", evals_dir, "documind-ai-YOUR-ID"):
    corpus += dc.chunk_document(d, "acme")
corpus_chars = sum(len(c["text"]) for c in corpus)
print(f"ACME corpus: {len(corpus)} chunks, {corpus_chars:,} characters, ~{corpus_chars // CHARS_PER_TOKEN:,} tokens, "
      f"{len(batches([c['text'] for c in corpus]))} embedding calls if batched together")

Expected: `4 calls; texts per call [85, 78, 78, 42]`, `1 call for 2 texts`, and `ACME corpus: 1628 chunks, 2,064,459 characters, ~688,153 tokens, 47 embedding calls`. The lane batches per document, so it makes a few more.

## 5. The message contract at the edge (Rs 0)

`IngestMessage` is the object record Cloud Storage publishes, as the worker chooses to see it. The tenant is the first folder of the path; a path that escapes its prefix, an object at the bucket root and a zero-byte object are refused with the field named - that last one is what `make poison` uploads.

In [ ]:
import json
from pydantic import ValidationError
from contracts import IngestMessage, DocumentContract, sha256_of

record = {"bucket": "documind-ai-YOUR-ID-uploads", "name": "acme/hr_policy_2026.md",
          "size": "160949", "contentType": "text/markdown", "generation": "1758400000000000"}
msg = IngestMessage.model_validate_json(json.dumps(record))
print("tenant:", msg.tenant_id, "| uri:", msg.gcs_uri, "| size:", msg.size, type(msg.size).__name__)

for label, bad in [("traversal", dict(record, name="../etc/passwd")),
                   ("no tenant prefix", dict(record, name="hr_policy_2026.md")),
                   ("zero bytes", dict(record, size="0"))]:
    try:
        IngestMessage.model_validate_json(json.dumps(bad))
        print(label, ": accepted (should not happen)")
    except ValidationError as e:
        print(f"{label:17}: refused - {e.errors()[0]['msg']}")

doc = DocumentContract(tenant_id=msg.tenant_id, sha256=sha256_of(bytes_v1), gcs_uri=msg.gcs_uri, pages=1)
print("doc_key    :", doc.doc_key)
print("chunk_id(1):", doc.chunk_id(1))
print("the notebook's id for the same text:", chunks_v1[1]["chunk_id"])

Expected: `tenant: acme`, the size coerced to `int`; three refusals - `object name escapes its prefix`, `object is not under a tenant prefix`, `Input should be greater than or equal to 1`; `doc_key acme_497809ff...` and `chunk_id(1) acme:497809ff...#1` beside the notebook's `acme:hr_policy_2026#NP-03`.

## 6. The sparse leg (Rs 0)

`shared/sparse_encoder.sparse_encode` is imported by `indexer.py` and by the API's `hybrid.py`: one deterministic space, `blake2b-tf-v1`, for the document side and the query side.

In [ ]:
from shared.sparse_encoder import sparse_encode, SPARSE_DIMS, SPARSE_ENCODER_VERSION

values, dims = sparse_encode(np03["text"])
print(SPARSE_ENCODER_VERSION, "| space:", f"{SPARSE_DIMS:,}", "| NP-03 dimensions:", len(dims),
      "| first:", dims[:4], values[:4], "| max weight:", max(values))
q_values, q_dims = sparse_encode("what is the notice period for a confirmed employee")
shared_dims = sorted(set(dims) & set(q_dims))
print("question dimensions:", len(q_dims), "| shared with NP-03:", len(shared_dims),
      "| dot product:", sum(values[dims.index(d)] * q_values[q_dims.index(d)] for d in shared_dims))

Expected: `NP-03 dimensions: 33`, `max weight: 2.5` (a token that appears four times: 1 + 3 x 0.5), and a handful of dimensions shared with the question - the words they have in common.

## 7. Three real vectors - COSTS MONEY

The only cell that calls Google. It embeds the two chunks the re-issue changed and PB-02 with `text-embedding-005` on a **regional** client (`us-central1`; embeddings are not served from `global`), asks for 768 dimensions and `RETRIEVAL_DOCUMENT` exactly as `embed_all` does, validates the vectors with a copy of `valid_vector` (indexer.py, line 64), and prices them. Text embeddings are listed per 1,000 input characters on the Vertex AI pricing page ("Embeddings for text", online requests); the rate below is that list price as read for this lesson - re-verify it on the live page before quoting it to anyone. Needs `PROJECT_ID`, the auth cell, and the Vertex AI API enabled.

In [ ]:
# COSTS MONEY - a fraction of a rupee. Needs PROJECT_ID set above, the auth cell run, and the Vertex AI API enabled.
import math
from google import genai

# services/ingest/indexer.py, lines 23-26 and 64 - copied
EMBEDDING_MODEL = "text-embedding-005"
EMBEDDING_TASK_TYPE = "RETRIEVAL_DOCUMENT"
EMBEDDING_DIMENSIONS = 768

def valid_vector(vector) -> bool:
    return (vector is not None and len(list(vector)) == EMBEDDING_DIMENSIONS
            and all(math.isfinite(v) for v in vector))

client = genai.Client(enterprise=True, project=PROJECT_ID, location="us-central1")   # embeddings are REGIONAL
texts = [chunks_v2[i]["text"] for i in misses_v2] + [chunks_v1[2]["text"]]           # the two changed chunks, and PB-02
response = client.models.embed_content(
    model=EMBEDDING_MODEL, contents=texts,
    config={"output_dimensionality": EMBEDDING_DIMENSIONS, "task_type": EMBEDDING_TASK_TYPE})
vectors = [list(e.values) for e in response.embeddings]
print("vectors:", len(vectors), "| dimensions:", sorted({len(v) for v in vectors}),
      "| all valid:", all(valid_vector(v) for v in vectors))

USD_INR = 85
USD_PER_1K_CHARS = 0.000025     # Vertex AI pricing page, "Embeddings for text", online - re-verify on the live page
chars = sum(len(t) for t in texts)
tokens = [getattr(getattr(e, "statistics", None), "token_count", None) for e in response.embeddings]
tokens = [int(t) if t else max(1, len(x) // 3) for t, x in zip(tokens, texts)]    # the API's count, or the kit's estimate
print("tokens per text:", tokens, "| characters:", chars)
usd = chars / 1000 * USD_PER_1K_CHARS
print(f"these three: ${usd:.6f} = Rs {usd * USD_INR:.4f}   (priced per character, not per token)")
handbook_chars = sum(len(c["text"]) for c in chunks_v1)
print(f"the whole handbook once: Rs {handbook_chars / 1000 * USD_PER_1K_CHARS * USD_INR:.2f}; "
      f"the ACME corpus once: Rs {corpus_chars / 1000 * USD_PER_1K_CHARS * USD_INR:.2f}")

Expected: `vectors: 3 | dimensions: [768] | all valid: True`, three token counts of roughly 60 to 80, and a cost of about Rs 0.001 for these three, Rs 0.34 for the handbook and Rs 4.4 for the ACME corpus - the embedding is the cheapest thing on the lane; Document AI's per-page charge on the PDFs is what a full corpus load costs (lesson 3.1).

What comes next: lesson 3.4 reads the functions that make these rows current - the claim, the swap, the retire and the undo - and its notebook reads a real ledger.